# 🏢 AI-Powered Business Research Assistant
### End-to-End Retrieval-Augmented Generation (RAG) Architecture

This notebook demonstrates the step-by-step pipeline powering the **AI-Powered Business Research Assistant**:
1. **Multi-Source Web Scraping & Cleaning**: Ingesting unstructured text from corporate websites and recent market news.
2. **Semantic Text Chunking**: Partitioning content using `RecursiveCharacterTextSplitter`.
3. **Dense Vector Embeddings & FAISS Indexing**: Generating high-dimensional representations and building a similarity index.
4. **Automated Executive Analysis**: Synthesizing Company Overview, Key Developments, Potential Business Challenges, and Strategic Solutions.
5. **Conversational Follow-up Q&A**: Question answering grounded strictly in retrieved chunks with direct source attribution.

## 1. Imports and Environment Setup

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_groq import ChatGroq

# Load environment variables from .env
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
print("Environment initialized. API Key available:", bool(groq_api_key))

## 2. Ingest and Clean Web Content
We scrape the company website and 1-3 market news articles using realistic user-agent headers to bypass 403 blocks, stripping non-content tags (scripts, styles, navigation, footer).

In [ ]:
def scrape_clean_url(url: str) -> Document:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript", "svg", "aside"]):
        tag.decompose()
        
    lines = [line.strip() for line in soup.get_text(separator="\n").splitlines() if line.strip()]
    cleaned_text = "\n".join(lines)[:40000]
    return Document(page_content=cleaned_text, metadata={"source": url})

# Target Inputs
company_name = "Tata Motors"
company_url = "https://www.tatamotors.com"
news_urls = [
    "https://www.moneycontrol.com/news/business/tata-motors-mahindra-gain-certificates-for-production-linked-payouts-11281691.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html",
    "https://www.moneycontrol.com/news/business/stocks/buy-tata-motors-target-of-rs-743-kr-choksey-11080811.html"
]

# Ingest sources
documents = [scrape_clean_url(company_url)] + [scrape_clean_url(u) for u in news_urls]
print(f"Successfully ingested {len(documents)} source documents.")

## 3. Semantic Document Chunking
We use `RecursiveCharacterTextSplitter` with 1000 character chunks and 150 character overlap to retain context across sentence boundaries.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " "],
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(documents)
print(f"Total semantic chunks generated: {len(chunks)}")

## 4. Dense Vector Embeddings & FAISS Vector Index
Embedding the chunks into an optimized vector space using `FastEmbed` and creating a similarity index with `FAISS`.

In [ ]:
embeddings = FastEmbedEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
print("FAISS vector store indexed successfully!")

## 5. Executive Business Intelligence Synthesis
We retrieve relevant chunks and prompt the LLM to generate the 5 structured sections:
1. **Company Overview**
2. **Key Developments**
3. **Potential Business Challenge** (cautious, objective framing)
4. **Recommended Solution Area** (Customer Analytics, Sales Analytics, Supply Chain Analytics, Process Automation, or Data Visualization)
5. **Evidence / Sources**

In [ ]:
query = f"{company_name} corporate overview core products recent developments financial performance challenges"
retrieved_chunks = vectorstore.similarity_search(query, k=6)
context_text = "\n\n---\n\n".join([c.page_content for c in retrieved_chunks])

# Initialize LLM via Groq
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.2, groq_api_key=groq_api_key)

analysis_prompt = f"""
You are an executive business research analyst. You have been provided with retrieved context from {company_name}'s official website and recent news articles.

Context:
{context_text}

Format your response in Markdown using EXACTLY these 5 sections:
### 🏢 Company Overview (2-3 concise sentences)
### 📰 Key Developments (3-5 bullet points)
### ⚠️ Potential Business Challenge (Infer 1 challenge using cautious, analytical phrasing)
### 💡 Recommended Solution (Select 1 capability: Customer Analytics, Sales Analytics, Supply Chain Analytics, Process Automation, or Data Visualization, and justify)
### 📚 Evidence / Sources (List supporting URLs)
"""

report = llm.invoke(analysis_prompt)
print(report.content)

## 6. Interactive Grounded Q&A
The user can ask follow-up questions about the company. Answers are retrieved from the FAISS index with cited sources.

In [ ]:
user_question = "Why did you identify this as a potential business challenge, and what evidence supports it?"
qa_chunks = vectorstore.similarity_search(user_question, k=4)
qa_context = "\n\n".join([f"Source: {c.metadata.get('source')}\n{c.page_content}" for c in qa_chunks])

qa_prompt = f"""
Answer the question based STRICTLY on the retrieved context below. State sources cited.

Context:
{qa_context}

Question:
{user_question}
"""

answer = llm.invoke(qa_prompt)
print(answer.content)